# Week 10 Lab — Machine Translation

## Learning Goals
- Understand the core challenges in **Machine Translation (MT)** and key approaches.
- Implement **BLEU score** from scratch and use `sacrebleu` for standardised evaluation.
- Explore additional MT metrics: **chrF**, **METEOR**, and **TER**.
- Use a pre-trained **Neural MT model** (MarianMT) from HuggingFace for translation.
- Analyse **common MT errors** and practise **post-editing**.
- Evaluate the effect of **domain** and **language pair** on translation quality.

## Part 0 — Setup

In [ ]:
!pip install -q sacrebleu transformers sentencepiece torch nltk

In [ ]:
import re
import math
import string
from collections import Counter

import nltk
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sacrebleu

nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)

print("All imports OK.")

---
## Part 1 — MT Fundamentals and Error Taxonomy

Machine Translation systems must handle a wide range of linguistic phenomena. Before writing any code, let us build intuition for the kinds of errors MT systems make.

### MT Error Types

| Error Type | Description | Example |
|------------|-------------|--------|
| **Omission** | A source word or phrase is missing from the translation | |
| **Addition** | Extra words appear in the translation not present in the source | |
| **Mistranslation** | Incorrect word choice or meaning | |
| **Word order** | Correct words but in wrong sequence | |
| **Punctuation** | Incorrect punctuation marks | |
| **Morphology** | Wrong inflection, tense, or agreement | |
| **Terminology** | Incorrect domain-specific term | |
| **Fluency** | Grammatically awkward though roughly correct | |

### Task 1A — Error Classification

For each MT output below (English source → Vietnamese MT output), identify the error type(s) using the taxonomy above.

| # | Source (EN) | MT Output (VI) | Error Type(s) | Correction |
|---|-------------|----------------|--------------|------------|
| 1 | The scientists published their findings last week. | Các nhà khoa học công bố phát hiện của họ. | | |
| 2 | She has been studying machine learning for three years. | Cô ấy đã học máy học trong ba năm. | | |
| 3 | The bank raised interest rates by 0.5 percent. | Ngân hàng tăng lãi suất lên 0,5 phần trăm. | | |
| 4 | He didn't go to the meeting because he was sick. | Anh ấy không đi họp vì anh ấy. | | |
| 5 | The results were surprisingly accurate. | Các kết quả rất chính xác. | | |

_Fill in the table above._

### Task 1B — MT System Comparison

Below are three MT outputs for the same source sentence. Rank them from best (1) to worst (3) and justify.

**Source (EN):** *"Despite heavy rain, the football match continued until the referee blew the final whistle."*

- **System A:** *"Mặc dù mưa to, trận bóng đá vẫn tiếp tục cho đến khi trọng tài thổi còi kết thúc."*
- **System B:** *"Mặc dù mưa nặng, trận đấu bóng đá tiếp tục cho đến còi cuối cùng của trọng tài thổi."*
- **System C:** *"Mặc dù trời mưa to, trận đấu bóng đá vẫn tiếp tục cho đến khi trọng tài thổi còi kết thúc trận đấu."*

_Your ranking and justification here._

### Questions
1. What makes MT **fundamentally harder** than tasks like sentiment classification? Name three linguistic phenomena that pose specific challenges.
2. What is the difference between **adequacy** (does the translation convey the meaning?) and **fluency** (is the translation grammatically natural?) in MT evaluation?
3. Why is **word-for-word (literal) translation** often incorrect? Give an example of an English idiom that should not be translated literally.

_Your answers here._

---
## Part 2 — BLEU Score from Scratch

**BLEU** (Bilingual Evaluation Understudy) is the most widely used automatic MT evaluation metric. It measures n-gram overlap between the system output (hypothesis) and one or more human translations (references).

$$\text{BLEU} = \text{BP} \times \exp\left(\sum_{n=1}^{N} w_n \log p_n\right)$$

where:
- $p_n$ is the **modified n-gram precision** for order $n$
- $w_n = 1/N$ (uniform weights, typically $N=4$)
- **BP** is the **brevity penalty** to discourage short outputs:

$$\text{BP} = \begin{cases} 1 & \text{if } c > r \\ e^{1 - r/c} & \text{if } c \leq r \end{cases}$$

where $c$ = hypothesis length and $r$ = reference length (shortest reference).

In [ ]:
# Coding Task A — Modified n-gram precision

def get_ngrams(tokens: list, n: int) -> Counter:
    """Return a Counter of all n-grams in `tokens`."""
    return Counter(tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1))


def modified_precision(hypothesis: list, references: list, n: int) -> float:
    """
    Compute modified n-gram precision.

    The "modified" part clips each hypothesis n-gram count to the maximum
    count of that n-gram across all references. This prevents the system
    from gaming BLEU by repeating high-frequency n-grams.

    Args:
        hypothesis  : list of tokens (one translated sentence)
        references  : list of list of tokens (one or more reference sentences)
        n           : n-gram order
    Returns:
        precision value in [0, 1], or 0 if no hypothesis n-grams
    """
    hyp_ngrams = get_ngrams(hypothesis, n)
    if not hyp_ngrams:
        return 0.0

    # For each hypothesis n-gram, find the max count across all references
    clipped_count = 0
    for ngram, hyp_count in hyp_ngrams.items():
        # TODO: find max_ref_count = max count of this ngram across all refs
        max_ref_count = max(
            get_ngrams(ref, n).get(ngram, 0) for ref in references
        )  # TODO
        # TODO: clip: take min(hyp_count, max_ref_count)
        clipped_count += min(hyp_count, max_ref_count)  # TODO

    # TODO: divide clipped count by total hypothesis n-gram count
    total_hyp_count = sum(hyp_ngrams.values())
    return clipped_count / total_hyp_count  # TODO


# Quick test
hyp  = "the the the the".split()
refs = [["the cat sat on the mat".split()]]
p1 = modified_precision(hyp, refs, n=1)
print(f"Modified 1-gram precision (degenerate): {p1:.3f}  (should be 0.500)")

hyp2  = "the cat sat on the mat".split()
p1_ok = modified_precision(hyp2, refs, n=1)
p2_ok = modified_precision(hyp2, refs, n=2)
print(f"Modified 1-gram precision (perfect): {p1_ok:.3f}  (should be 1.000)")
print(f"Modified 2-gram precision (perfect): {p2_ok:.3f}  (should be 1.000)")

In [ ]:
# Coding Task B — Brevity Penalty

def brevity_penalty(hypothesis_length: int, reference_length: int) -> float:
    """
    Compute the BLEU brevity penalty.

    Args:
        hypothesis_length : number of tokens in the hypothesis
        reference_length  : effective reference length (shortest ref length)
    Returns:
        BP in (0, 1]
    """
    if hypothesis_length == 0:
        return 0.0
    # TODO: if hypothesis longer than reference, BP = 1
    # otherwise BP = exp(1 - reference_length / hypothesis_length)
    if hypothesis_length > reference_length:
        return 1.0  # TODO
    return math.exp(1 - reference_length / hypothesis_length)  # TODO


# Verify
print(f"BP (hyp=10, ref=10): {brevity_penalty(10, 10):.4f}  (expect 1.0000)")
print(f"BP (hyp=10, ref=15): {brevity_penalty(10, 15):.4f}  (expect {math.exp(1-15/10):.4f})")
print(f"BP (hyp=15, ref=10): {brevity_penalty(15, 10):.4f}  (expect 1.0000)")

In [ ]:
# Coding Task C — Corpus BLEU (sentence-level and corpus-level)

def sentence_bleu(hypothesis: str, references: list, max_n: int = 4,
                  smooth: float = 1.0) -> float:
    """
    Compute sentence-level BLEU with add-`smooth` smoothing for zero n-gram counts.

    Args:
        hypothesis  : hypothesis string (will be lowercased and tokenised on whitespace)
        references  : list of reference strings
        max_n       : maximum n-gram order (default 4)
        smooth      : smoothing count added to numerator and denominator when p_n=0
    Returns:
        BLEU score in [0, 1]
    """
    hyp_tokens  = hypothesis.lower().split()
    ref_tokens  = [r.lower().split() for r in references]

    # Brevity penalty uses the closest reference length
    ref_lengths = [len(r) for r in ref_tokens]
    closest_ref_len = min(ref_lengths, key=lambda l: abs(l - len(hyp_tokens)))
    bp = brevity_penalty(len(hyp_tokens), closest_ref_len)

    # Compute weighted log-sum of precisions
    log_sum = 0.0
    for n in range(1, max_n + 1):
        pn = modified_precision(hyp_tokens, ref_tokens, n)
        # TODO: apply smoothing: if pn == 0, use smooth / (denom + smooth)
        # Simple approach: add smooth to numerator counts
        if pn == 0:
            pn = smooth / (len(hyp_tokens) - n + 1 + smooth)  # TODO
        log_sum += (1.0 / max_n) * math.log(pn)  # TODO: add weighted log(pn)

    # TODO: return BP * exp(log_sum)
    return bp * math.exp(log_sum)  # TODO


def corpus_bleu(hypotheses: list, references_list: list, max_n: int = 4) -> float:
    """
    Corpus-level BLEU: aggregate n-gram counts across all sentences, then compute.

    Args:
        hypotheses      : list of hypothesis strings
        references_list : list of lists of reference strings (one inner list per hyp)
        max_n           : maximum n-gram order
    Returns:
        corpus BLEU in [0, 1]
    """
    # Aggregate clipped counts and total hypothesis n-gram counts across sentences
    clipped_counts = [0] * max_n
    total_counts   = [0] * max_n
    hyp_len, ref_len = 0, 0

    for hyp_str, refs in zip(hypotheses, references_list):
        hyp_toks = hyp_str.lower().split()
        ref_toks = [r.lower().split() for r in refs]
        hyp_len += len(hyp_toks)
        closest = min([len(r) for r in ref_toks],
                      key=lambda l: abs(l - len(hyp_toks)))
        ref_len += closest

        for n in range(1, max_n + 1):
            hyp_ng = get_ngrams(hyp_toks, n)
            total_counts[n-1] += sum(hyp_ng.values())
            for ngram, cnt in hyp_ng.items():
                max_ref = max(get_ngrams(r, n).get(ngram, 0) for r in ref_toks)
                clipped_counts[n-1] += min(cnt, max_ref)

    bp = brevity_penalty(hyp_len, ref_len)
    log_sum = 0.0
    for n in range(max_n):
        if clipped_counts[n] == 0 or total_counts[n] == 0:
            log_sum += float('-inf')
            break
        # TODO: compute log(clipped / total) and add (1/max_n) * log
        log_sum += (1.0 / max_n) * math.log(clipped_counts[n] / total_counts[n])  # TODO

    return bp * math.exp(log_sum) if log_sum != float('-inf') else 0.0


# --- Test data ---
HYP_1  = "the cat is on the mat"
REF_1A = "the cat sat on the mat"
REF_1B = "there is a cat on the mat"

bleu_1 = sentence_bleu(HYP_1, [REF_1A, REF_1B])
print(f"Sentence BLEU (example 1): {bleu_1:.4f}")

HYP_2  = "a quick brown fox"
REF_2  = "the quick brown fox jumps over the lazy dog"
bleu_2 = sentence_bleu(HYP_2, [REF_2])
print(f"Sentence BLEU (example 2): {bleu_2:.4f}")

In [ ]:
# Coding Task D — Validate against sacrebleu

# A small English→French MT evaluation set
# Sources are English sentences; hypotheses are MT outputs; references are human translations
EN_FR_DATA = [
    {
        "source":    "The European Parliament adopted the resolution by a large majority.",
        "hypothesis": "Le Parlement européen a adopté la résolution à une grande majorité.",
        "reference":  "Le Parlement européen a adopté la résolution par une large majorité.",
    },
    {
        "source":    "Climate change poses an existential threat to future generations.",
        "hypothesis": "Le changement climatique pose une menace existentielle aux générations futures.",
        "reference":  "Le changement climatique représente une menace existentielle pour les générations futures.",
    },
    {
        "source":    "Scientists have discovered a new species of deep-sea fish.",
        "hypothesis": "Les scientifiques ont découvert une nouvelle espèce de poisson des profondeurs.",
        "reference":  "Des scientifiques ont découvert une nouvelle espèce de poisson des eaux profondes.",
    },
    {
        "source":    "The stock market fell sharply after the announcement.",
        "hypothesis": "Le marché boursier a chuté fortement après l'annonce.",
        "reference":  "La bourse a fortement chuté après l'annonce.",
    },
    {
        "source":    "She graduated with honours from the University of Paris.",
        "hypothesis": "Elle a obtenu son diplôme avec mentions de l'Université de Paris.",
        "reference":  "Elle est diplômée avec mention de l'Université de Paris.",
    },
]

hypotheses = [d["hypothesis"] for d in EN_FR_DATA]
references = [[d["reference"]] for d in EN_FR_DATA]

# Our implementation
our_corpus_bleu = corpus_bleu(hypotheses, references)

# sacrebleu implementation
sb_bleu = sacrebleu.corpus_bleu(hypotheses, [[d["reference"] for d in EN_FR_DATA]])

print(f"Our corpus BLEU   : {our_corpus_bleu * 100:.2f}")
print(f"sacrebleu BLEU    : {sb_bleu.score:.2f}")
print()
print("Per-sentence BLEU:")
print(f"{'Source':<55} {'Our':>6} {'SB':>6}")
print("-" * 70)
for d in EN_FR_DATA:
    our = sentence_bleu(d["hypothesis"], [d["reference"]]) * 100
    sb  = sacrebleu.sentence_bleu(d["hypothesis"], [d["reference"]]).score
    print(f"{d['source'][:54]:<55} {our:>6.2f} {sb:>6.2f}")

### Questions
1. Why is **modified** n-gram precision used in BLEU instead of standard precision? Give a concrete example where standard precision would be misleading.
2. BLEU operates at **corpus level** — why can per-sentence BLEU scores be unreliable?
3. A translation receives BLEU = 0 even though it conveys the correct meaning. How is this possible? Give an example.
4. What does the **brevity penalty** prevent and why is it necessary?

_Your answers here._

---
## Part 3 — Additional MT Metrics

BLEU is widely criticised for correlating poorly with human judgement, especially at the sentence level. Several alternative metrics address its limitations.

- **chrF** (Character F-score): operates on character n-grams rather than word n-grams, making it more robust to morphological variation.
- **TER** (Translation Edit Rate): the minimum number of edits (insert, delete, substitute, shift) needed to transform the hypothesis into the reference, normalised by reference length.
- **METEOR**: aligns hypothesis and reference using exact match, stemming, and synonym matching before computing F-score.

In [ ]:
# Coding Task A — chrF score using sacrebleu

# Compute chrF for the EN→FR data
chrf = sacrebleu.corpus_chrf(hypotheses, [[d["reference"] for d in EN_FR_DATA]])
print(f"Corpus chrF: {chrf.score:.2f}")

# Per-sentence
print("\nPer-sentence chrF vs BLEU:")
print(f"{'Source':<50} {'BLEU':>6} {'chrF':>6}")
print("-" * 65)
for d in EN_FR_DATA:
    b  = sacrebleu.sentence_bleu(d["hypothesis"], [d["reference"]]).score
    c  = sacrebleu.sentence_chrf(d["hypothesis"], [d["reference"]]).score
    print(f"{d['source'][:49]:<50} {b:>6.2f} {c:>6.2f}")

In [ ]:
# Coding Task B — Translation Edit Rate (TER) from scratch

def ter_score(hypothesis: str, reference: str) -> float:
    """
    Compute TER (without the shift operation — simplified version).

    TER = edit_distance(hyp_tokens, ref_tokens) / len(ref_tokens)

    The edit distance here counts insertions, deletions, and substitutions
    (no shifts). This is the Levenshtein distance at word level.

    Args:
        hypothesis : hypothesis string
        reference  : reference string
    Returns:
        TER score (lower is better; 0 = perfect, 1 = many edits)
    """
    hyp = hypothesis.lower().split()
    ref = reference.lower().split()

    if not ref:
        return 0.0 if not hyp else float('inf')

    # TODO: implement word-level Levenshtein distance (dynamic programming)
    # dp[i][j] = edit distance between hyp[:i] and ref[:j]
    m, n = len(hyp), len(ref)
    dp = [[0] * (n + 1) for _ in range(m + 1)]

    for i in range(m + 1):
        dp[i][0] = i  # TODO: cost of deleting i hypothesis words
    for j in range(n + 1):
        dp[0][j] = j  # TODO: cost of inserting j reference words

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if hyp[i-1] == ref[j-1]:
                dp[i][j] = dp[i-1][j-1]  # TODO: no edit needed
            else:
                dp[i][j] = 1 + min(
                    dp[i-1][j],    # deletion
                    dp[i][j-1],    # insertion
                    dp[i-1][j-1]   # substitution
                )  # TODO

    return dp[m][n] / n  # TODO: normalise by reference length


# Test
print("TER examples:")
test_pairs = [
    ("the cat sat on the mat", "the cat sat on the mat"),   # perfect → 0
    ("a cat sat on mat",       "the cat sat on the mat"),   # 2 edits
    ("dog barked loudly",      "the cat sat on the mat"),   # very different
]
for hyp, ref in test_pairs:
    t = ter_score(hyp, ref)
    print(f"  TER={t:.3f}  hyp='{hyp}'  ref='{ref}'")

# EN→FR data
print("\nPer-sentence TER (EN→FR):")
for d in EN_FR_DATA:
    t = ter_score(d["hypothesis"], d["reference"])
    b = sacrebleu.sentence_bleu(d["hypothesis"], [d["reference"]]).score
    print(f"  TER={t:.3f}  BLEU={b:.2f}  src='{d['source'][:50]}...'")  

In [ ]:
# Coding Task C — Metric Comparison Visualisation

# Compute all metrics for a degraded hypothesis set
# We simulate increasingly poor translations by progressively shuffling words

base_hyp = "Le Parlement européen a adopté la résolution par une large majorité."
base_ref = "Le Parlement européen a adopté la résolution par une large majorité."

def degrade_translation(sentence: str, n_swaps: int, seed: int = 42) -> str:
    """Randomly swap n_swaps pairs of adjacent words in the sentence."""
    words = sentence.split()
    rng = __import__('random').Random(seed)
    for _ in range(n_swaps):
        i = rng.randint(0, len(words) - 2)
        words[i], words[i+1] = words[i+1], words[i]
    return " ".join(words)

swap_levels = list(range(0, 7))
bleu_scores = []
chrf_scores = []
ter_scores  = []

for n in swap_levels:
    degraded = degrade_translation(base_hyp, n)
    bleu_scores.append(sacrebleu.sentence_bleu(degraded, [base_ref]).score)
    chrf_scores.append(sacrebleu.sentence_chrf(degraded, [base_ref]).score)
    ter_scores.append(ter_score(degraded, base_ref) * 100)  # scale to 0–100

fig, ax = plt.subplots(figsize=(8, 4))
# TODO: plot BLEU, chrF, TER (inverted so higher=better) vs swap_levels
ax.plot(swap_levels, bleu_scores, marker='o', label='BLEU')      # TODO
ax.plot(swap_levels, chrf_scores, marker='s', label='chrF')      # TODO
ax.plot(swap_levels, [100 - t for t in ter_scores], marker='^', label='1 - TER (×100)')  # TODO
ax.set_xlabel("Number of word swaps (degradation level)")
ax.set_ylabel("Score")
ax.set_title("MT Metrics Under Progressive Word-Order Degradation")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("mt_metrics_degradation.png", dpi=100)
plt.show()

### Questions
1. Why does **chrF** tend to be more robust than BLEU for morphologically rich languages (e.g. Finnish, Turkish, Vietnamese)?
2. A hypothesis has low BLEU but high chrF. What type of translation pattern could explain this?
3. What does a **TER > 1.0** mean in practice?
4. Why is it important to use **multiple evaluation metrics** rather than relying solely on BLEU?

_Your answers here._

---
## Part 4 — Neural Machine Translation with MarianMT

**MarianMT** is a family of pre-trained neural MT models available through HuggingFace Transformers. Each model covers a specific language pair (e.g. `Helsinki-NLP/opus-mt-en-fr` for English→French).

In [ ]:
# Coding Task A — Load MarianMT and translate EN → FR

from transformers import MarianMTModel, MarianTokenizer

MODEL_NAME = "Helsinki-NLP/opus-mt-en-fr"

# TODO: load the tokeniser and model
tokenizer = MarianTokenizer.from_pretrained(MODEL_NAME)  # TODO
model     = MarianMTModel.from_pretrained(MODEL_NAME)     # TODO

print(f"Model loaded: {MODEL_NAME}")
print(f"Vocab size   : {tokenizer.vocab_size}")

In [ ]:
# Coding Task B — Translation function with beam search

def translate(texts: list, tokenizer, model, num_beams: int = 4,
              max_length: int = 128) -> list:
    """
    Translate a list of source texts using beam search.

    Args:
        texts      : list of source strings
        tokenizer  : MarianTokenizer
        model      : MarianMTModel
        num_beams  : beam size (1 = greedy)
        max_length : maximum generated sequence length
    Returns:
        list of translated strings
    """
    # TODO:
    # 1. tokenise inputs (padding=True, truncation=True, return_tensors='pt')
    # 2. call model.generate() with num_beams and max_length
    # 3. decode with tokenizer.batch_decode(skip_special_tokens=True)
    inputs = tokenizer(
        texts, return_tensors="pt", padding=True, truncation=True
    )  # TODO
    translated = model.generate(
        **inputs, num_beams=num_beams, max_length=max_length
    )  # TODO
    return tokenizer.batch_decode(translated, skip_special_tokens=True)  # TODO


# Translate the evaluation set
sources = [d["source"] for d in EN_FR_DATA]
marian_hypotheses = translate(sources, tokenizer, model)

print("MarianMT EN → FR translations:")
print(f"{'Source':<55} {'Translation'}")
print("-" * 100)
for src, hyp in zip(sources, marian_hypotheses):
    print(f"{src[:54]:<55} {hyp}")

In [ ]:
# Coding Task C — Evaluate MarianMT with BLEU and chrF

ref_list = [d["reference"] for d in EN_FR_DATA]

marian_bleu = sacrebleu.corpus_bleu(marian_hypotheses, [ref_list])
marian_chrf = sacrebleu.corpus_chrf(marian_hypotheses, [ref_list])
marian_ter  = sum(ter_score(h, r) for h, r in zip(marian_hypotheses, ref_list)) / len(ref_list)

# Compare with the provided (imperfect) hypotheses from Part 2
orig_bleu = sacrebleu.corpus_bleu(hypotheses, [ref_list])
orig_chrf = sacrebleu.corpus_chrf(hypotheses, [ref_list])
orig_ter  = sum(ter_score(h, r) for h, r in zip(hypotheses, ref_list)) / len(ref_list)

print(f"{'System':<20} {'BLEU':>8} {'chrF':>8} {'TER':>8}")
print("-" * 48)
print(f"{'Original hyp':<20} {orig_bleu.score:>8.2f} {orig_chrf.score:>8.2f} {orig_ter:>8.3f}")
print(f"{'MarianMT':<20} {marian_bleu.score:>8.2f} {marian_chrf.score:>8.2f} {marian_ter:>8.3f}")

In [ ]:
# Coding Task D — Beam size experiment

beam_sizes = [1, 2, 4, 8]
bleu_by_beam = []
time_by_beam = []

import time
for beam in beam_sizes:
    start = time.time()
    hyps = translate(sources, tokenizer, model, num_beams=beam)
    elapsed = time.time() - start
    bleu = sacrebleu.corpus_bleu(hyps, [ref_list]).score
    bleu_by_beam.append(bleu)
    time_by_beam.append(elapsed)
    print(f"  Beams={beam}: BLEU={bleu:.2f}, time={elapsed:.2f}s")

fig, ax1 = plt.subplots(figsize=(7, 4))
ax2 = ax1.twinx()
# TODO: plot BLEU (left axis) and time (right axis) vs beam_sizes
ax1.plot(beam_sizes, bleu_by_beam, marker='o', color='steelblue', label='BLEU')  # TODO
ax2.plot(beam_sizes, time_by_beam, marker='s', color='tomato',    label='Time (s)')  # TODO
ax1.set_xlabel("Beam size")
ax1.set_ylabel("BLEU", color='steelblue')
ax2.set_ylabel("Time (s)", color='tomato')
ax1.set_title("BLEU vs. Beam Size (EN→FR)")
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='lower right')
plt.tight_layout()
plt.savefig("bleu_vs_beam.png", dpi=100)
plt.show()

### Questions
1. What is **beam search** and why does it generally outperform greedy decoding for MT? What is its main computational drawback?
2. At what beam size do you observe diminishing BLEU returns in your experiment? How does this trade off against latency?
3. MarianMT uses a **sequence-to-sequence Transformer** with an encoder-decoder architecture. Explain what the encoder and decoder each learn to do.
4. What would you expect to happen to BLEU if you used a **back-translation** strategy to create more training data?

_Your answers here._

---
## Part 5 — Post-Editing Analysis

**Post-editing** (PE) is the process of correcting MT output to professional quality. It is widely used in industry (human-in-the-loop MT). Analysing post-editing effort helps estimate MT system quality beyond automatic metrics.

In [ ]:
# Dataset: MT output and human post-edited versions
PE_DATA = [
    {
        "source":    "The committee approved the budget unanimously.",
        "mt_output": "Le comité a approuvé le budget à l'unanimité.",
        "post_edit": "Le comité a approuvé le budget à l'unanimité.",
    },
    {
        "source":    "She submitted her thesis three days before the deadline.",
        "mt_output": "Elle a soumis sa thèse trois jours avant la date limite.",
        "post_edit": "Elle a rendu sa thèse trois jours avant la date limite.",
    },
    {
        "source":    "The construction of the new bridge will take approximately two years.",
        "mt_output": "La construction du nouveau pont prendra approximativement deux ans.",
        "post_edit": "La construction du nouveau pont durera environ deux ans.",
    },
    {
        "source":    "Investors are concerned about rising inflation and interest rates.",
        "mt_output": "Les investisseurs s'inquiètent de l'inflation croissante et des taux d'intérêt.",
        "post_edit": "Les investisseurs sont préoccupés par la hausse de l'inflation et des taux d'intérêt.",
    },
    {
        "source":    "The patient was prescribed antibiotics for seven days.",
        "mt_output": "Le patient a reçu une prescription d'antibiotiques pour sept jours.",
        "post_edit": "Le patient s'est vu prescrire des antibiotiques pendant sept jours.",
    },
]

print(f"Post-editing dataset: {len(PE_DATA)} segments")

In [ ]:
# Coding Task A — Post-editing effort with TER

print("Post-editing Effort (TER: MT→PE):")
print(f"{'Source':<50} {'TER (effort)':>14}")
print("-" * 70)

ter_efforts = []
for d in PE_DATA:
    effort = ter_score(d["mt_output"], d["post_edit"])
    ter_efforts.append(effort)
    print(f"{d['source'][:49]:<50} {effort:>14.3f}")

print(f"\nMean post-editing TER: {sum(ter_efforts)/len(ter_efforts):.3f}")
print("(0 = no edits needed, higher = more effort)")

In [ ]:
# Coding Task B — Compute word-level diff (insertions, deletions, substitutions)

def word_diff(mt: str, pe: str) -> dict:
    """
    Compute word-level edit operations between MT output and post-edited version.
    Returns dict: {'insertions': int, 'deletions': int, 'substitutions': int, 'total_edits': int}

    Use the edit-distance DP from ter_score to recover the alignment,
    then count operation types.
    """
    a = mt.lower().split()
    b = pe.lower().split()
    m, n = len(a), len(b)

    # Build DP table
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1): dp[i][0] = i
    for j in range(n + 1): dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if a[i-1] == b[j-1]:
                dp[i][j] = dp[i-1][j-1]
            else:
                dp[i][j] = 1 + min(dp[i-1][j], dp[i][j-1], dp[i-1][j-1])

    # Traceback to count operation types
    insertions = deletions = substitutions = 0
    i, j = m, n
    while i > 0 or j > 0:
        if i > 0 and j > 0 and a[i-1] == b[j-1]:
            i -= 1; j -= 1  # match — no edit
        elif i > 0 and j > 0 and dp[i][j] == dp[i-1][j-1] + 1:
            substitutions += 1; i -= 1; j -= 1  # TODO: substitution
        elif j > 0 and dp[i][j] == dp[i][j-1] + 1:
            insertions += 1; j -= 1             # TODO: insertion into MT
        else:
            deletions += 1; i -= 1             # TODO: deletion from MT

    return {
        "insertions":   insertions,
        "deletions":    deletions,
        "substitutions": substitutions,
        "total_edits":  insertions + deletions + substitutions,
    }


print("Word-level edit analysis:")
print(f"{'Seg':<5} {'Ins':>5} {'Del':>5} {'Sub':>5} {'Total':>7}")
print("-" * 35)
for i, d in enumerate(PE_DATA, 1):
    diff = word_diff(d["mt_output"], d["post_edit"])
    print(f"{i:<5} {diff['insertions']:>5} {diff['deletions']:>5} {diff['substitutions']:>5} {diff['total_edits']:>7}")

In [ ]:
# Coding Task C — Your own post-editing task

# Below are five MT outputs that contain errors. Post-edit them into correct French.
# After editing, compute TER and classify the error types you fixed.

YOUR_PE_TASK = [
    {
        "source":    "The train departed from Paris at seven in the morning.",
        "mt_output": "Le train est parti de Paris à sept dans le matin.",
        "your_edit": "???"  # TODO: write your corrected translation
    },
    {
        "source":    "He failed the exam because he did not study enough.",
        "mt_output": "Il a échoué l'examen parce qu'il n'a pas étudié assez.",
        "your_edit": "???"  # TODO
    },
    {
        "source":    "The company launched its new product in three markets simultaneously.",
        "mt_output": "La société a lancé son nouveau produit dans trois marchés simultanément.",
        "your_edit": "???"  # TODO
    },
    {
        "source":    "Research shows that sleep deprivation affects cognitive performance.",
        "mt_output": "La recherche montre que la privation de sommeil affecte la performance cognitive.",
        "your_edit": "???"  # TODO
    },
    {
        "source":    "The minister resigned following the corruption scandal.",
        "mt_output": "Le ministre a résigné suivant le scandale de corruption.",
        "your_edit": "???"  # TODO
    },
]

# Once you've filled in your_edit, run this cell to measure your post-editing effort
print("Your post-editing effort:")
for d in YOUR_PE_TASK:
    if d["your_edit"] != "???":
        effort = ter_score(d["mt_output"], d["your_edit"])
        print(f"  TER={effort:.3f}  '{d['source'][:50]}'")
    else:
        print(f"  [Not edited] '{d['source'][:50]}'")

### Questions
1. What is the difference between **light post-editing** (fixing only critical errors) and **full post-editing** (achieving publication quality)?
2. Why is TER considered a better proxy for post-editing effort than BLEU?
3. Which error types (from Part 1's taxonomy) did you encounter most frequently in Task 5C? Why might those errors be systematic?

_Your answers here._

---
## Part 6 — Domain Adaptation and Low-Resource MT

Pre-trained MT models are typically trained on general-domain parallel data (e.g. news, European Parliament proceedings). Performance drops significantly in specialised domains (medical, legal, technical) or for low-resource language pairs.

In [ ]:
# Coding Task A — Domain gap analysis

# General-domain sentences (in-domain for opus-mt-en-fr)
GENERAL_DOMAIN = [
    {"source": "The government announced new tax reforms yesterday.",
     "reference": "Le gouvernement a annoncé hier de nouvelles réformes fiscales."},
    {"source": "The football team won the championship for the third consecutive year.",
     "reference": "L'équipe de football a remporté le championnat pour la troisième année consécutive."},
    {"source": "She decided to travel to Europe during the summer holidays.",
     "reference": "Elle a décidé de voyager en Europe pendant les vacances d'été."},
]

# Medical domain (out-of-domain for opus-mt-en-fr)
MEDICAL_DOMAIN = [
    {"source": "The patient presented with acute myocardial infarction and was given thrombolysis.",
     "reference": "Le patient présentait un infarctus du myocarde aigu et a reçu une thrombolyse."},
    {"source": "Contraindications include hypersensitivity to the active substance or excipients.",
     "reference": "Les contre-indications comprennent l'hypersensibilité à la substance active ou aux excipients."},
    {"source": "The biopsy revealed metastatic carcinoma with lymph node involvement.",
     "reference": "La biopsie a révélé un carcinome métastatique avec atteinte ganglionnaire."},
]

# Translate both domains
gen_hyps = translate([d["source"] for d in GENERAL_DOMAIN], tokenizer, model)
med_hyps = translate([d["source"] for d in MEDICAL_DOMAIN], tokenizer, model)

gen_refs = [d["reference"] for d in GENERAL_DOMAIN]
med_refs = [d["reference"] for d in MEDICAL_DOMAIN]

gen_bleu = sacrebleu.corpus_bleu(gen_hyps, [gen_refs]).score
med_bleu = sacrebleu.corpus_bleu(med_hyps, [med_refs]).score
gen_chrf = sacrebleu.corpus_chrf(gen_hyps, [gen_refs]).score
med_chrf = sacrebleu.corpus_chrf(med_hyps, [med_refs]).score

print(f"{'Domain':<12} {'BLEU':>8} {'chrF':>8}")
print("-" * 30)
print(f"{'General':<12} {gen_bleu:>8.2f} {gen_chrf:>8.2f}")
print(f"{'Medical':<12} {med_bleu:>8.2f} {med_chrf:>8.2f}")

print("\nMedical translations (inspect for terminology errors):")
for d, hyp in zip(MEDICAL_DOMAIN, med_hyps):
    print(f"  src: {d['source']}")
    print(f"  hyp: {hyp}")
    print(f"  ref: {d['reference']}")
    print()

In [ ]:
# Coding Task B — Lexical coverage analysis

def oov_rate(text: str, tokenizer) -> float:
    """
    Compute the fraction of whitespace-tokenised source words that are split
    into more than one subword token by the MarianMT tokeniser.
    A high rate suggests the text contains many words unknown to the model.

    Args:
        text      : input string
        tokenizer : MarianTokenizer
    Returns:
        fraction of words that are split (0 = all known, 1 = all OOV)
    """
    words = text.split()
    if not words:
        return 0.0
    n_split = 0
    for word in words:
        # TODO: tokenise just this word and check if it produces > 1 token
        tokens = tokenizer.tokenize(word)
        if len(tokens) > 1:
            n_split += 1  # TODO
    return n_split / len(words)  # TODO


print("OOV (subword split) rate by domain:")
print(f"{'Text':<55} {'OOV rate':>10}")
print("-" * 68)
for d in GENERAL_DOMAIN:
    rate = oov_rate(d["source"], tokenizer)
    print(f"{d['source'][:54]:<55} {rate:>10.3f}")
print()
for d in MEDICAL_DOMAIN:
    rate = oov_rate(d["source"], tokenizer)
    print(f"{d['source'][:54]:<55} {rate:>10.3f}")

### Questions
1. Why does MT quality drop in the **medical domain** compared to the general domain? Name two types of domain-specific linguistic challenge.
2. What is **domain adaptation** for NMT? Describe two adaptation strategies and their trade-offs.
3. What is **back-translation** and why is it particularly useful for low-resource language pairs?
4. Explain the concept of **zero-shot cross-lingual transfer** in multilingual MT models (e.g. NLLB, mBART). When would you use this instead of a dedicated bilingual model?

_Your answers here._

---
## Part 7 — MT Ethics and Human Evaluation (Discussion)

### 7.1 Gender Bias in MT

Many MT systems exhibit gender bias, assigning default masculine forms to gender-neutral source words.

**Example (EN→FR):**
- Source: *"The doctor examined the patient and the nurse took notes."*
- Biased output: *"Le médecin a examiné le patient et l'infirmier a pris des notes."* (male nurse)
- Fair output: *"Le médecin a examiné le patient et l'infirmière a pris des notes."*

**Task:** Translate the following English sentences using MarianMT. Note any gender biases you observe, and explain how they arise from training data.

```python
gender_test_sentences = [
    "The engineer solved the problem quickly.",
    "The nurse cared for the elderly patients.",
    "The CEO announced record profits.",
    "The teacher explained the lesson carefully.",
    "The programmer fixed the critical bug.",
]
```

_Record your observations below._

---

### 7.2 Human Evaluation

Automatic metrics like BLEU have well-documented limitations. Human evaluation can use:
- **Adequacy rating** (1–5 scale): does the translation convey the full meaning?
- **Fluency rating** (1–5 scale): is the translation grammatically natural?
- **Direct Assessment (DA)**: raters score how well the MT output expresses the source meaning on a continuous scale.
- **Multidimensional Quality Metrics (MQM)**: annotate specific error types and weights.

**Task:** For the five medical domain translations from Part 6A, manually rate each on:
- Adequacy (1 = meaning lost, 5 = full meaning preserved)
- Fluency (1 = unnatural, 5 = perfectly natural)

_Fill in the table:_

| # | Source | Adequacy | Fluency | Notes |
|---|--------|----------|---------|-------|
| 1 | The patient presented with... | | | |
| 2 | Contraindications include... | | | |
| 3 | The biopsy revealed... | | | |

---

### 7.3 MT in High-Stakes Domains

1. Identify **two risks** of deploying unvalidated MT systems in medical or legal contexts.
2. What safeguards would you recommend before deploying MT in a hospital?
3. Why is **back-translation of patient records** without human review potentially dangerous?

_Your answers here._

In [ ]:
# Optional: gender bias experiment from Section 7.1

gender_test_sentences = [
    "The engineer solved the problem quickly.",
    "The nurse cared for the elderly patients.",
    "The CEO announced record profits.",
    "The teacher explained the lesson carefully.",
    "The programmer fixed the critical bug.",
]

gender_translations = translate(gender_test_sentences, tokenizer, model)

print("Gender bias test (EN → FR):")
for src, hyp in zip(gender_test_sentences, gender_translations):
    print(f"  EN: {src}")
    print(f"  FR: {hyp}")
    print()

---
## Submission

- Complete all `TODO` sections in the code cells.
- Answer all written questions in the markdown cells above.
- Include in your submission:
  - The **BLEU / chrF / TER comparison table** from Part 2D
  - The **metric degradation plot** from Part 3C (`mt_metrics_degradation.png`)
  - The **beam size vs BLEU plot** from Part 4D (`bleu_vs_beam.png`)
  - The **domain gap table** from Part 6A
  - Your **post-edited translations** from Part 5C
  - Your **gender bias observations** from Part 7.1
- Submit your completed `.ipynb` file as instructed.